# Urban Heat & Cooling-Priority Mapping — Track B / SB2: Interactive Labeling Tool

**NUS-ISS Practice Module, Week 2.** Companion notebook to `generate_validation_sample.ipynb`
(SB1). Run SB1 first — this notebook loads its output and lets you and your
teammate label all 200 points together on an interactive map.

**Capabilities:**
1. Shows all 200 points on a map, colored by labeling status (orange = not yet
   labeled, green = labeled).
2. **Resume work** — `load_default()` (SB2.2) auto-resumes from your last save,
   or `upload_and_load()` lets you upload a CSV from your computer instead.
3. **Save progress** — every label you confirm auto-saves to Drive immediately
   via `save_progress()`, so nothing is lost if the session ends early. A
   manual "Save all progress now" button is also provided.
4. **Export** — an "Export final CSV+GeoJSON" button in the panel (SB2.4) calls
   `export_final()`, writing the exact schema `generate_validation_sample.ipynb`
   produced, ready for Step 3-4 (RF/U-Net/ensemble evaluation). Safe to click
   anytime as a checkpoint, not just once at the end.
5. **Click a point → pick a label.** Click any marker on the map; a panel below
   the map shows that point's ID and WorldCover reference class (context only,
   never the answer), and lets you choose the agreed label, confidence, and
   notes, then save.

**Design note:** loading, saving, and exporting are all defined as plain
functions in one cell (SB2.2) — `load_default()`, `upload_and_load()`,
`save_progress()`, `export_final()`, `download_backup()` — so you don't need
to hunt across separate cells; the panel's buttons just call these directly.

**Basemap:** Esri World Imagery — Esri's continuously-updated *current*
satellite layer, matching the "label against recent imagery, not WorldCover's
2021 snapshot" decision from the labeling guide.

**Colab note:** `ipyleaflet` needs the custom widget manager enabled in Colab
(handled in Setup below). If markers don't render, re-run the Setup cells and
then Runtime → Restart runtime once.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [7]:
# --- SETUP CELL 1: Install dependencies -------------------------------------
!pip install -q ipyleaflet ipywidgets pandas geopandas


## Setup 2 — Enable widgets, mount Drive

In [8]:
# --- SETUP CELL 2: Enable Colab widget manager + mount Drive ----------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
else:
    print("Not running in Colab — Drive mount skipped, use local paths in SB2.1.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


---
# SB2 — Interactive labeling


## SB2.1 — Config (paths, class options — matches the labeling guide's Section 2)

In [9]:
# --- SB2 CELL 1: Config ------------------------------------------------------
EXPORT_FOLDER = "urban_heat_sg"
EXPORT_FILE_PREFIX = "validation_sample_200"

DRIVE_DIR = f"/content/drive/MyDrive/{EXPORT_FOLDER}" if IN_COLAB else "."
SB1_OUTPUT_CSV = f"{DRIVE_DIR}/{EXPORT_FILE_PREFIX}.csv"          # SB1's export
WORK_CSV = f"{DRIVE_DIR}/{EXPORT_FILE_PREFIX}_labeling_progress.csv"  # auto-saved progress
FINAL_CSV = f"{DRIVE_DIR}/{EXPORT_FILE_PREFIX}_labeled.csv"       # final export (SB2.6)
FINAL_GEOJSON = f"{DRIVE_DIR}/{EXPORT_FILE_PREFIX}_labeled.geojson"

# Class options — CUT DOWN to the 4-class scheme from the gate review:
# vegetation / built_up / bare / water. Plain-English text shown in the
# dropdown, stored VALUE (second item in each tuple) is the schema string
# used everywhere downstream (evaluation script, guide). "uncertain" stays
# as an escape hatch, not one of the 4 real classes.
CLASS_OPTIONS = [
    ("-- select a label --", ""),
    ("🌿 Vegetation — trees, shrubs, grass, crops, mangroves", "vegetation"),
    ("🏢 Built-up — buildings, roads, pavement, construction sites", "built_up"),
    ("🟤 Bare — exposed soil, sand, cleared land", "bare"),
    ("💧 Water — pond/reservoir/drain edge, mixed pixel, or misclassified point", "water"),
    ("❓ Uncertain — flag for team review", "uncertain_flag_for_review"),
]

# Auto-migration map: any label saved under the OLD 8-class schema (from
# before the gate-review cutdown) gets collapsed to the new 4-class scheme
# automatically on load — see collapse_label() in SB2.2. No manual re-work
# needed for points already labeled under the old scheme.
COLLAPSE_MAP = {
    "tree_cover": "vegetation",
    "shrubland": "vegetation",
    "grassland": "vegetation",
    "cropland": "vegetation",
    "mangroves": "vegetation",
    "herbaceous_wetland": "vegetation",
    "moss_lichen": "vegetation",
    "bare_sparse_veg": "bare",
    "snow_ice": "bare",
    "built_up": "built_up",
    "water": "water",
}

# Human-readable version of the WorldCover reference class shown in the
# panel (SB2.4) — shows the collapsed 4-class bucket plus the original
# fine-grained WorldCover class in parentheses, for context. This is
# display only — the underlying worldcover_class_name column in the CSV
# is unchanged (still the original fine-grained WorldCover label).
_WC_BUCKET_EMOJI = {"vegetation": "🌿", "built_up": "🏢", "bare": "🟤", "water": "💧"}

def wc_display(worldcover_class_name):
    bucket = COLLAPSE_MAP.get(worldcover_class_name, worldcover_class_name)
    emoji = _WC_BUCKET_EMOJI.get(bucket, "❓")
    return f"{emoji} {bucket} (WorldCover: {worldcover_class_name})"

# Confidence scale — plain-English text, values stay as short codes for the CSV.
# Still an open decision per the guide's Section 6 checklist — edit the wording
# below once you and your teammate agree on precise definitions.
CONFIDENCE_OPTIONS = [
    ("-- select confidence --", ""),
    ("😕 Low — genuinely unsure, borderline case", "1-low"),
    ("🙂 Medium — fairly confident, minor doubt", "2-medium"),
    ("😄 High — clearly obvious, no doubt", "3-high"),
]

REQUIRED_COLUMNS = [
    "point_id", "lon", "lat", "worldcover_class", "worldcover_class_name",
    "agreed_label", "confidence", "notes",
]

# Fallback sample generation — used only if SB1's output isn't found (see
# SB2.2). No Earth Engine needed: fetches the real Singapore boundary via
# data.gov.sg and draws plain random points within it (NOT stratified by
# WorldCover). If you later run generate_validation_sample.ipynb (SB1)
# properly, it overwrites SB1_OUTPUT_CSV with a real stratified sample —
# any labeling progress you've already saved to WORK_CSV is unaffected
# either way, since it's a separate file.
TOTAL_POINTS = 200
RANDOM_SEED = 42
SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"  # MP19 Subzone Boundary (No Sea)

print("Config set.")
print("SB1 output (source):", SB1_OUTPUT_CSV)
print("Progress autosave:  ", WORK_CSV)
print("Final export:       ", FINAL_CSV, "/", FINAL_GEOJSON)


Config set.
SB1 output (source): /content/drive/MyDrive/urban_heat_sg/validation_sample_200.csv
Progress autosave:   /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeling_progress.csv
Final export:        /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeled.csv / /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeled.geojson


## SB2.2 — Load points + save/export functions (all in one cell)

Defines everything you need for loading and saving in a single place, then
loads immediately:
- `load_default()` — resumes from previous progress if it exists, else loads
  SB1's fresh output if that exists, **else generates a fallback sample right
  here** (no need to run another notebook first). The fallback fetches the
  real Singapore boundary and draws plain random points within it — it's
  NOT stratified by WorldCover vegetation class the way SB1's sample is, so
  swap in SB1's real output later if you want that rigor; it overwrites the
  same file automatically.
- **Auto-migration to the 4-class scheme.** The labeling schema was cut down
  to 4 classes (vegetation / built_up / bare / water) per the gate review.
  Any `agreed_label` already saved under the old 8-class schema (tree_cover,
  shrubland, grassland, etc.) is automatically collapsed to the matching
  4-class bucket the moment the file loads — no manual re-labeling needed.
  See `COLLAPSE_MAP` in SB2.1 for the exact mapping.
- `upload_and_load()` — alternative: upload a CSV from your computer instead
  (uncomment the line below to use this instead of `load_default()`).
- `save_progress()` — writes the current in-memory table to `WORK_CSV`. Called
  automatically by the Save buttons in SB2.4, but you can also call it directly
  any time.
- `export_final()` — writes the final CSV + GeoJSON to `FINAL_CSV`/`FINAL_GEOJSON`,
  in the schema `generate_validation_sample.ipynb` (SB1) produced. Call this
  whenever you want a checkpoint or once labeling is complete — safe to call
  repeatedly, it just overwrites.
- `download_backup()` — (Colab only) triggers a browser download of `WORK_CSV`,
  if you want a local copy outside Drive.


In [10]:
# --- SB2 CELL 2: Load + save + export functions --------------------------------
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os


def _validate_columns(df):
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Loaded CSV is missing required columns: {missing}")
    for c in ["agreed_label", "confidence", "notes"]:
        df[c] = df[c].fillna("").astype(str)

    # Auto-migrate any labels saved under the old 8-class schema to the
    # new 4-class scheme (vegetation/built_up/bare/water) — no manual
    # re-labeling needed for points already done before the cutdown.
    valid_new_values = {v for _, v in CLASS_OPTIONS}
    n_migrated = 0
    for idx, val in df["agreed_label"].items():
        if val and val not in valid_new_values and val in COLLAPSE_MAP:
            df.at[idx, "agreed_label"] = COLLAPSE_MAP[val]
            n_migrated += 1
    if n_migrated > 0:
        print(f"Auto-migrated {n_migrated} label(s) from the old 8-class scheme to the new "
              f"4-class scheme (vegetation/built_up/bare/water).")

    # Safety check: flag any label that's neither a valid new-scheme value
    # NOR something COLLAPSE_MAP knew how to migrate. These are silent-
    # corruption risks — the map dropdown won't recognize them, and if you
    # click that point and hit Save without noticing, the real label gets
    # overwritten with blank. Surfacing them loudly here instead.
    unrecognized = df[
        (df["agreed_label"].str.strip() != "") & (~df["agreed_label"].isin(valid_new_values))
    ]
    if len(unrecognized) > 0:
        print(f"⚠️  {len(unrecognized)} row(s) have an agreed_label the notebook doesn't ")
        print("   recognize (not in the current 4-class scheme, not in COLLAPSE_MAP either).")
        print("   These will show as blank in the map panel — do NOT click 'Save label' on")
        print("   these points without first checking what the original value should map to,")
        print("   or you will overwrite it with blank. Affected point_ids:")
        print("  ", list(unrecognized["point_id"]))

    return df


def generate_fallback_sample():
    """Self-contained fallback: no SB1 output found, so generate a fresh
    sample right here. No Earth Engine needed — fetches the real Singapore
    boundary via data.gov.sg and draws plain random points within it.
    NOT stratified by WorldCover class (that needs Earth Engine / SB1) —
    flagged clearly below and in the output. Saves the result to
    SB1_OUTPUT_CSV so this only needs to happen once."""
    import requests
    import json as _json
    import numpy as np
    from shapely.geometry import shape, Point
    from shapely.ops import unary_union

    print("⚠️  No SB1 output found — generating a fallback sample here instead.")
    print("   This sample is PLAIN RANDOM within Singapore's real boundary,")
    print("   NOT stratified by WorldCover vegetation class (that needs")
    print("   Earth Engine — run generate_validation_sample.ipynb for a properly")
    print("   stratified sample; it will overwrite this file when you do).")
    print()

    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{SUBZONE_DATASET_ID}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    geojson_bytes = requests.get(payload["data"]["url"]).content
    subzone_geojson = _json.loads(geojson_bytes)

    geoms = [shape(feat["geometry"]) for feat in subzone_geojson["features"]]
    sg_boundary = unary_union(geoms)
    print(f"Fetched Singapore boundary ({len(geoms)} subzones dissolved).")

    rng = np.random.default_rng(RANDOM_SEED)
    minx, miny, maxx, maxy = sg_boundary.bounds
    rows = []
    attempts = 0
    while len(rows) < TOTAL_POINTS and attempts < TOTAL_POINTS * 200:
        attempts += 1
        x, y = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        if sg_boundary.contains(Point(x, y)):
            rows.append({
                "point_id": f"P{len(rows) + 1:04d}",
                "lon": x, "lat": y,
                "worldcover_class": -1,
                "worldcover_class_name": "unknown",
                "agreed_label": "", "confidence": "", "notes": "",
            })

    df = pd.DataFrame(rows)
    os.makedirs(DRIVE_DIR, exist_ok=True)
    df.to_csv(SB1_OUTPUT_CSV, index=False)
    print(f"Generated {len(df)} points, saved to {SB1_OUTPUT_CSV}")
    return _validate_columns(df)


def load_default():
    """Load previous progress if it exists, else fall back to SB1's output,
    else generate a fallback sample automatically (no manual steps needed)."""
    if os.path.exists(WORK_CSV):
        df = pd.read_csv(WORK_CSV)
        print(f"Resumed from previous progress: {WORK_CSV} ({len(df)} rows)")
    elif os.path.exists(SB1_OUTPUT_CSV):
        df = pd.read_csv(SB1_OUTPUT_CSV)
        print(f"Loaded fresh sample from SB1: {SB1_OUTPUT_CSV} ({len(df)} rows)")
    else:
        df = generate_fallback_sample()
    return _validate_columns(df)


def upload_and_load():
    """Upload a CSV from your computer (e.g. a locally-saved backup) to resume from it."""
    if not IN_COLAB:
        raise RuntimeError("File upload widget requires Colab. Use load_default() with a local path instead.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(fname)
    print(f"Uploaded and loaded: {fname} ({len(df)} rows)")
    return _validate_columns(df)


def save_progress():
    """Write the current in-memory table to WORK_CSV. Safe to call repeatedly."""
    points_df.to_csv(WORK_CSV, index=False)


def export_final():
    """Write final CSV + GeoJSON in SB1's schema. Warns (doesn't block) if
    some points are still unlabeled — safe to call as a checkpoint anytime."""
    n_unlabeled = (points_df["agreed_label"].str.strip() == "").sum()
    if n_unlabeled > 0:
        print(f"⚠️  {n_unlabeled} point(s) still unlabeled. Exporting anyway — "
              f"call export_final() again after finishing for a complete export.")
    else:
        print("✅ All points labeled.")

    points_df.to_csv(FINAL_CSV, index=False)
    gdf = gpd.GeoDataFrame(
        points_df,
        geometry=[Point(xy) for xy in zip(points_df["lon"], points_df["lat"])],
        crs="EPSG:4326",
    )
    gdf.to_file(FINAL_GEOJSON, driver="GeoJSON")
    print(f"Exported:\n  {FINAL_CSV}\n  {FINAL_GEOJSON}")
    print("\nLabel distribution:")
    print(points_df["agreed_label"].value_counts())


def download_backup():
    """(Colab only) Trigger a browser download of the current progress file."""
    if IN_COLAB:
        from google.colab import files
        save_progress()
        files.download(WORK_CSV)
    else:
        print(f"Not in Colab — file already at {WORK_CSV}")


# --- Choose ONE of the following, then run this cell: -----------------------
points_df = load_default()
# points_df = upload_and_load()   # <-- uncomment instead, to resume from an uploaded file

n_labeled = (points_df["agreed_label"].str.strip() != "").sum()
print(f"Progress: {n_labeled}/{len(points_df)} points already labeled.")
points_df.head()


Resumed from previous progress: /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeling_progress.csv (200 rows)
Progress: 120/200 points already labeled.


,point_id,lon,lat,worldcover_class,worldcover_class_name,agreed_label,confidence,notes
0,P0001,103.973166,1.404011,-1,unknown,,,
1,P0002,103.667552,1.299253,-1,unknown,,,
2,P0003,103.784715,1.447920,-1,unknown,,,
3,P0004,103.916547,1.415463,-1,unknown,,,
4,P0005,104.005266,1.355826,-1,unknown,,,


## SB2.3 — Build the map + labeling state

Esri World Imagery basemap (current/recent satellite imagery). Markers:
orange = not yet labeled, green = labeled. Click a marker to select it for
labeling in SB2.4's panel.


In [11]:
# --- SB2 CELL 3: Map + state --------------------------------------------------
from ipyleaflet import Map, basemaps, CircleMarker, LayerGroup
import ipywidgets as widgets
from IPython.display import display

current_idx = None  # index into points_df of the currently-selected point

sg_center = (1.3521, 103.8198)
m = Map(basemap=basemaps.Esri.WorldImagery, center=sg_center, zoom=11, scroll_wheel_zoom=True)
marker_group = LayerGroup()
m.add_layer(marker_group)

status_out = widgets.Output()


def marker_color(row):
    return "green" if str(row["agreed_label"]).strip() else "orange"


def make_click_handler(idx):
    def _handler(**kwargs):
        select_point(idx)
    return _handler


def refresh_markers():
    """Rebuild all markers from points_df's current state (colors reflect
    labeling progress). Called after every save so progress is visible live."""
    new_markers = []
    for idx, row in points_df.iterrows():
        cm = CircleMarker(
            location=(row["lat"], row["lon"]),
            radius=6,
            color=marker_color(row),
            fill_color=marker_color(row),
            fill_opacity=0.85,
            weight=2,
        )
        cm.on_click(make_click_handler(idx))
        new_markers.append(cm)
    marker_group.layers = tuple(new_markers)


refresh_markers()
print(f"Map ready with {len(points_df)} points. Click a marker, then use the panel in SB2.4.")


Map ready with 200 points. Click a marker, then use the panel in SB2.4.


## SB2.4 — Labeling panel (select a point's class, confidence, notes → save)

Run this cell once — it displays the map plus the control panel below it.
Clicking a marker updates the panel to that point; the dropdowns pre-fill
with whatever was already saved for that point (if you're resuming).


In [12]:
# --- SB2 CELL 4: Labeling panel ------------------------------------------------
point_id_label = widgets.HTML(value="<b>No point selected — click a marker on the map above.</b>")
wc_reference_label = widgets.HTML(value="")

label_dropdown = widgets.Dropdown(options=CLASS_OPTIONS, description="Label:")
confidence_dropdown = widgets.Dropdown(options=CONFIDENCE_OPTIONS, description="Confidence:")
notes_box = widgets.Textarea(value="", description="Notes:", layout=widgets.Layout(width="400px"))

save_button = widgets.Button(description="Save label", button_style="success")
save_all_button = widgets.Button(description="Save all progress now", button_style="info")
export_button = widgets.Button(description="Export final CSV+GeoJSON", button_style="primary")
next_unlabeled_button = widgets.Button(description="Jump to next unlabeled point", button_style="warning")

progress_label = widgets.HTML(value="")


def update_progress_label():
    n_labeled = (points_df["agreed_label"].str.strip() != "").sum()
    progress_label.value = f"<b>Progress: {n_labeled}/{len(points_df)} labeled</b>"


def select_point(idx):
    global current_idx
    current_idx = idx
    row = points_df.loc[idx]
    point_id_label.value = f"<b>Selected: {row['point_id']}</b> (lon={row['lon']:.5f}, lat={row['lat']:.5f})"
    wc_display_text = wc_display(row["worldcover_class_name"])
    wc_reference_label.value = (
        f"<i>WorldCover reference (context only, NOT the answer): {wc_display_text}</i>"
    )
    class_values = [v for _, v in CLASS_OPTIONS]
    confidence_values = [v for _, v in CONFIDENCE_OPTIONS]
    label_dropdown.value = row["agreed_label"] if row["agreed_label"] in class_values else ""
    confidence_dropdown.value = row["confidence"] if row["confidence"] in confidence_values else ""
    notes_box.value = row["notes"] if isinstance(row["notes"], str) else ""
    m.center = (row["lat"], row["lon"])
    m.zoom = 18


def on_save_clicked(b):
    global current_idx
    if current_idx is None:
        with status_out:
            status_out.clear_output()
            print("No point selected — click a marker first.")
        return
    points_df.loc[current_idx, "agreed_label"] = label_dropdown.value
    points_df.loc[current_idx, "confidence"] = confidence_dropdown.value
    points_df.loc[current_idx, "notes"] = notes_box.value
    refresh_markers()
    update_progress_label()
    save_progress()
    with status_out:
        status_out.clear_output()
        print(f"Saved {points_df.loc[current_idx, 'point_id']} -> "
              f"'{label_dropdown.value}'. Autosaved to {WORK_CSV}")


def on_save_all_clicked(b):
    save_progress()
    with status_out:
        status_out.clear_output()
        print(f"All progress saved to {WORK_CSV}")


def on_export_clicked(b):
    with status_out:
        status_out.clear_output()
        export_final()


def on_next_unlabeled_clicked(b):
    unlabeled = points_df.index[points_df["agreed_label"].str.strip() == ""]
    if len(unlabeled) == 0:
        with status_out:
            status_out.clear_output()
            print("All points labeled!")
        return
    start_after = current_idx if current_idx is not None else -1
    candidates = [i for i in unlabeled if i > start_after]
    next_idx = candidates[0] if candidates else unlabeled[0]
    select_point(next_idx)


save_button.on_click(on_save_clicked)
save_all_button.on_click(on_save_all_clicked)
export_button.on_click(on_export_clicked)
next_unlabeled_button.on_click(on_next_unlabeled_clicked)

update_progress_label()

panel = widgets.VBox([
    progress_label,
    point_id_label,
    wc_reference_label,
    label_dropdown,
    confidence_dropdown,
    notes_box,
    widgets.HBox([save_button, next_unlabeled_button, save_all_button, export_button]),
    status_out,
])

display(m)
display(panel)


Map(center=[1.3521, 103.8198], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…